<a href="https://colab.research.google.com/github/kuds/courtside-dynamics/blob/main/notebooks/paddle_tennis_campaign.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PaddleTennis Staged Campaign

One notebook execution runs the whole from-scratch campaign without
stopping and starting:

1. **Gate leg** — train from scratch (the LS1 shape) and score the
   best checkpoint in-process with the behavioral diagnosis instrument
   against the frozen LS1 bars
   (`docs/paddle_tennis_registered_run_prereg_20260816.md` §1a).
2. **Auto-branch** — LS1 PASS continues the from-scratch lineage by
   warm-starting the main leg from the gate leg's own best; FAIL
   warm-starts from the frozen reference lineage instead. The session
   never idles waiting for a human decision.
3. **Main leg** — trains to the branch's frozen budget and writes a
   record-only reading of the registered §3 bands.

Every leg, verdict, and branch decision lands in
`campaign_manifest.json` under the campaign root (on Drive when
`USE_DRIVE = True`). **Re-running all cells resumes**: completed legs
are skipped, a half-trained attempt is left in place as evidence (a
fresh `attempt_NN` starts), and a recorded branch decision is reused
rather than re-decided.

**Doctrine note.** The settings cell *is* the campaign's frozen
protocol — the manifest fingerprints it, and a resumed execution with
silently changed budgets/bars refuses to run. For a *registered*
attempt, freeze the plan in a pre-registration doc first (the standing
convention); this notebook then merely executes it. The generic
single-run driver remains `sb3_training.ipynb`.

## 1. Install

In [ ]:
# Git branch, tag, or commit SHA to install. Keep `main` for normal runs;
# set this to an unmerged PR branch when smoke-testing changes.
REPO_REF = "main"

!pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/courtside-dynamics@{REPO_REF}"
print(f"Installed courtside-dynamics from ref: {REPO_REF}")

## 2. Configure Colab GPU

Runs before anything imports the environments so MuJoCo's EGL rendering is bootstrapped first (the humanoid notebook's proven ordering).

In [ ]:
from courtside_dynamics.colab_setup import setup_colab
setup_colab()

## 3. Campaign settings — the frozen plan

Everything numeric here ships pinned to the pre-registered values (prereg §1a/§1b/§2). `CAMPAIGN_ID` is the resume handle; the decision knobs (`FORCE_BRANCH`, `GATE_MIDDLE_ACTION`) are recorded in the manifest whenever they act. The warm-start and `ENV_KWARGS` knobs ship at their backward-compatible defaults (leg 1 from scratch, temperatures transferred, no sha pins, recipe task) and are fingerprinted like everything else.

In [ ]:
# One campaign == one CAMPAIGN_ID. The first execution creates
# <training_runs>/PaddleTennisCampaign/sac/<CAMPAIGN_ID>/; running the
# notebook again with the same ID RESUMES that campaign (completed
# legs are skipped, the recorded branch decision is reused). Start a
# new campaign by choosing a new ID.
CAMPAIGN_ID = "campaign_001"

USE_DRIVE = True
SEED = 0  # master seed; the main leg derives SEED + 10_000 (stage convention)
QUICK_TEST = False  # smoke the loop in minutes; not gate evidence (see FORCE_BRANCH)
DISCONNECT_WHEN_DONE = True

# --- The frozen stage plan ------------------------------------------
# Values are the pre-registered ones
# (docs/paddle_tennis_registered_run_prereg_20260816.md §1a/§1b/§2).
# Gate leg: the LS1 from-scratch shape.
GATE_TIMESTEPS = 1_000_000
# Main leg budget by branch: on PASS the campaign continues from the
# gate leg's own best (+2M -- the 1M already spent counts toward the
# 3M total); on FAIL the frozen warm-started shape runs at its full 3M.
MAIN_TIMESTEPS_ON_PASS = 2_000_000
MAIN_TIMESTEPS_ON_FALLBACK = 3_000_000
N_ENVS = 4
EVAL_FREQ = 25_000
CHECKPOINT_FREQ = 100_000
# Warm-started legs only (the §2 pairing): learning_starts on a fresh
# replay buffer holds the pretrained policy still through the first
# refill. From-scratch legs use the stack default.
WARM_START_LEARNING_STARTS = 25_000

# --- Warm-start knobs (per leg) -------------------------------------
# Leg 1 trains from scratch by default (the LS1 gate shape). Setting
# LEG1_WARM_START_RUN_DIR instead runs leg 1 warm-started -- e.g. the
# LT1 temperature-skip pilot shape
# (docs/paddle_tennis_lt1_prereg_20260823.md §2) under the campaign's
# manifest/resume machinery. The knobs mirror WarmStartConfig:
# TRANSFER_LOG_ENT_COEF = False is the temperature-skip warm start,
# and EXPECTED_ARTIFACT_SHA256 pins the source artifacts (name ->
# full sha256 or >= 8-char lowercase-hex prefix) so a moved artifact
# aborts the launch instead of voiding the pairing silently.
LEG1_WARM_START_RUN_DIR = None  # None = from scratch (the gate shape)
LEG1_TRANSFER_LOG_ENT_COEF = True
LEG1_EXPECTED_ARTIFACT_SHA256 = None
# Leg 2 always warm-starts from the branch-decided lineage; its knobs
# ship at the historical defaults (temperature transferred, no pins).
LEG2_TRANSFER_LOG_ENT_COEF = True
LEG2_EXPECTED_ARTIFACT_SHA256 = None

# --- Environment overrides ------------------------------------------
# Constructor kwargs layered over the recipe's task defaults for BOTH
# legs (training and evaluation environments alike -- the run-config
# [env]-table semantics, below the recipe's evaluation overrides).
# {} = the recipe defaults, byte-for-byte.
ENV_KWARGS = {}

# --- The gate (LS1 bars, frozen 2026-08-16) -------------------------
# Scored on the gate leg's best checkpoint by the same diagnosis
# instrument the run's own 100k probes use (30 episodes, calibration
# seeds 5200+). "At some checkpoint" bars are read conservatively at
# the best checkpoint: a PASS is sound, a FAIL may under-credit.
GATE_EPISODES = 30
GATE_SEED_START = 5200
GATE_BARS = {
    # LS-C: contact bootstraps (touched-after-bounce).
    "LS-C": {
        "metric": "touched_after_bounce_rate",
        "pass_at": 0.10,
        "fail_at": 0.05,
        "higher_is_better": True,
        "gating": True,
    },
    # LS-K1: a stroke forms (k=1 receiving survival).
    "LS-K1": {
        "metric": "k1_receiving_survival",
        "pass_at": 0.10,
        "fail_at": 0.02,
        "higher_is_better": True,
        "gating": True,
    },
    # LS-G: the reach gradient is live (informational, non-gating).
    "LS-G": {
        "metric": "ready_error_mean",
        "pass_at": 2.0,
        "fail_at": 2.4,
        "higher_is_better": False,
        "gating": False,
    },
}

# --- The branch rule ------------------------------------------------
# FAIL branch lineage: the frozen reference warm-start both pilots
# used. None stops the campaign at a gate FAIL instead (the
# from-scratch attempt is booked and nothing improvises a lineage).
FALLBACK_WARM_START_RUN_DIR = (
    "/content/drive/MyDrive/Finding Theta/courtside-dynamics/"
    "training_runs/PaddleTennis/sac/20260809_211147"
)
# A declared MIDDLE is a maintainer's call by doctrine: "stop" ends
# the execution for a human read; "continue"/"fallback" pre-declare
# the call.
GATE_MIDDLE_ACTION = "stop"
# Explicit human override of the gate ("continue" | "fallback"),
# recorded as forced in the manifest. Also how a stopped campaign is
# resumed after the maintainer decides a MIDDLE. Required (non-None)
# under QUICK_TEST, whose gate numbers are not evidence.
FORCE_BRANCH = None

# --- Final report (registered §3 bands, record-only) ----------------
# Single-checkpoint readings of the registered criteria on the main
# leg's best checkpoint. The registered verdict reads the full
# checkpoint series plus the held-out gate; this report is the
# in-notebook summary, never the booking.
FINAL_REPORT_BARS = {
    "RK1": {
        "metric": "k2_either_survival",
        "pass_at": 0.05,
        "fail_at": 0.01,
        "higher_is_better": True,
        "gating": False,
    },
    "RE1": {
        "metric": "touched_after_bounce_rate",
        "pass_at": 0.50,
        "fail_at": 0.41,
        "higher_is_better": True,
        "gating": False,
    },
    "RE3": {
        "metric": "k1_receiving_survival",
        "pass_at": 0.80,
        "fail_at": 0.68,
        "higher_is_better": True,
        "gating": False,
    },
    "RS2": {
        "metric": "k1_serving_survival",
        "pass_at": 0.25,
        "fail_at": 0.005,
        "higher_is_better": True,
        "gating": False,
    },
}

if FORCE_BRANCH not in (None, "continue", "fallback"):
    raise ValueError("FORCE_BRANCH must be None, 'continue', or 'fallback'.")
if GATE_MIDDLE_ACTION not in ("stop", "continue", "fallback"):
    raise ValueError("GATE_MIDDLE_ACTION must be stop|continue|fallback.")
if QUICK_TEST and FORCE_BRANCH is None:
    raise ValueError(
        "QUICK_TEST gate numbers are not branch evidence; set "
        "FORCE_BRANCH to smoke the full loop."
    )

## 4. Mount Drive, resolve the campaign root, detect resume

The campaign root is stable across executions — `training_runs/PaddleTennisCampaign/sac/<CAMPAIGN_ID>/` — so a disconnected session picks up where it left off. A fingerprint check refuses to resume under changed settings.

In [ ]:
from pathlib import Path

from courtside_dynamics.notebook_utils import (
    load_campaign_manifest,
    mount_drive,
    require_campaign_fingerprint,
    resolve_run_dir,
    write_campaign_manifest,
)

if USE_DRIVE:
    mount_drive()

# Stable (non-timestamped) root: the timestamp leaf is exactly what
# defeats resume, so the campaign root is
# <training_runs>/PaddleTennisCampaign/sac/<CAMPAIGN_ID> and
# CAMPAIGN_ID carries the identity.
campaign_parent = resolve_run_dir(
    "PaddleTennisCampaign", "sac", use_drive=USE_DRIVE, timestamp=False
)
CAMPAIGN_ROOT = Path(campaign_parent) / CAMPAIGN_ID
CAMPAIGN_ROOT.mkdir(parents=True, exist_ok=True)

FINGERPRINT = {
    "campaign_id": CAMPAIGN_ID,
    "seed": SEED,
    "quick_test": QUICK_TEST,
    "gate_timesteps": GATE_TIMESTEPS,
    "main_timesteps_on_pass": MAIN_TIMESTEPS_ON_PASS,
    "main_timesteps_on_fallback": MAIN_TIMESTEPS_ON_FALLBACK,
    "n_envs": N_ENVS,
    "eval_freq": EVAL_FREQ,
    "checkpoint_freq": CHECKPOINT_FREQ,
    "warm_start_learning_starts": WARM_START_LEARNING_STARTS,
    "leg1_warm_start_run_dir": LEG1_WARM_START_RUN_DIR,
    "leg1_transfer_log_ent_coef": LEG1_TRANSFER_LOG_ENT_COEF,
    "leg1_expected_artifact_sha256": LEG1_EXPECTED_ARTIFACT_SHA256,
    "leg2_transfer_log_ent_coef": LEG2_TRANSFER_LOG_ENT_COEF,
    "leg2_expected_artifact_sha256": LEG2_EXPECTED_ARTIFACT_SHA256,
    "env_kwargs": ENV_KWARGS,
    "gate_episodes": GATE_EPISODES,
    "gate_seed_start": GATE_SEED_START,
    "gate_bars": GATE_BARS,
    "fallback_warm_start_run_dir": FALLBACK_WARM_START_RUN_DIR,
}

manifest = load_campaign_manifest(CAMPAIGN_ROOT)
if manifest is None:
    manifest = {
        "campaign_id": CAMPAIGN_ID,
        "status": "running",
        "fingerprint": FINGERPRINT,
        "stages": {},
    }
    write_campaign_manifest(CAMPAIGN_ROOT, manifest)
    print(f"New campaign: {CAMPAIGN_ROOT}")
else:
    require_campaign_fingerprint(manifest, FINGERPRINT)
    print(f"Resuming campaign: {CAMPAIGN_ROOT} (status {manifest['status']})")
    for name, record in manifest["stages"].items():
        print(f"  {name}: {record['status']} -> {record.get('run_dir')}")
    if manifest.get("branch") is not None:
        print(f"  recorded branch decision: {manifest['branch']}")

## 5. Leg helpers

Thin glue over `notebook_utils`: build one leg's `TrainConfig` (the warm-started legs merge `learning_starts` into the recipe's calibrated SAC bundle), train it in a fresh `attempt_NN` dir, validate the run's `config.json` against the frozen plan (the prereg §6 check), and record the result in the manifest.

In [ ]:
from courtside_dynamics.notebook_utils import (
    next_stage_attempt_dir,
    print_stage_summary,
    resolve_warm_start_branch,
    score_paddle_stage,
    validate_run_config_against_plan,
)
from courtside_dynamics.recipes import (
    RECIPES,
    build_train_config,
    make_env_fn,
    make_eval_env_fn,
)
from courtside_dynamics.training import WarmStartConfig, train
from courtside_dynamics.training.artifacts import locate_artifact

RECIPE = "PaddleTennis"
GATE_STAGE = "leg1_scratch_gate"
MAIN_STAGE = "leg2_main"


def make_leg_config(*, log_dir, seed, total_timesteps, warm_start=None):
    # Explicit overrides replace recipe values wholesale, so the
    # warm-started legs hand build_train_config the recipe's calibrated
    # SAC bundle with learning_starts merged in -- never a bare
    # {"learning_starts": ...}, which would silently drop use_sde /
    # auto-entropy / train_freq. Under QUICK_TEST the frozen cadence and
    # budget are not pinned either, so the quick-test presets can
    # shrink the run.
    overrides = {"eval_verbose": 1}
    if not QUICK_TEST:
        overrides["n_envs"] = N_ENVS
        overrides["eval_freq"] = EVAL_FREQ
        overrides["checkpoint_freq"] = CHECKPOINT_FREQ
    if ENV_KWARGS:
        # The run-config [env]-table route: the same kwargs reach the
        # training env and sit below the recipe's evaluation overrides
        # for the eval env, so a physics tweak cannot split the two.
        overrides["env_fn"] = make_env_fn(RECIPE, env_overrides=ENV_KWARGS)
        overrides["eval_env_fn"] = make_eval_env_fn(
            RECIPE, base_env_overrides=ENV_KWARGS
        )
    if warm_start is not None:
        overrides["warm_start"] = warm_start
        overrides["model_kwargs"] = {
            **RECIPES[RECIPE].extra_cfg["model_kwargs"],
            "learning_starts": WARM_START_LEARNING_STARTS,
        }
    cfg = build_train_config(
        RECIPE,
        log_dir=str(log_dir),
        total_timesteps=None if QUICK_TEST else total_timesteps,
        quick_test=QUICK_TEST,
        seed=seed,
        **overrides,
    )
    if cfg.eval_freq > cfg.total_timesteps:
        raise ValueError("each leg's budget must include at least one evaluation")
    if ENV_KWARGS:
        # Fail a typo'd env kwarg here, in seconds (the config-file
        # route's own eager probe), instead of mid-train().
        for factory in (cfg.env_fn, cfg.eval_env_fn):
            if factory is not None:
                factory().close()
    source = warm_start.source_run_dir if warm_start is not None else None
    print(
        f"leg config: steps={cfg.total_timesteps:,} seed={cfg.seed} "
        f"n_envs={cfg.n_envs} eval_freq={cfg.eval_freq:,} "
        f"checkpoint_freq={cfg.checkpoint_freq:,} "
        f"warm_start={source or 'from scratch'}"
    )
    return cfg


def leg_expected_plan(*, seed, total_timesteps, warm_start):
    # The frozen-settings expectation for one leg's config.json (the
    # prereg §6 validation). QUICK_TEST hands budget/cadence to the
    # quick-test presets, so only a real run pins them to the plan.
    expected = {
        "seed": seed,
        "env_class": "PaddleTennisEnv",
        "env_kwargs": dict(ENV_KWARGS),
        "warm_start": None,
    }
    if not QUICK_TEST:
        expected["total_timesteps"] = total_timesteps
        expected["n_envs"] = N_ENVS
        expected["eval_freq"] = EVAL_FREQ
        expected["checkpoint_freq"] = CHECKPOINT_FREQ
    if warm_start is not None:
        # Suffix, not the full path: the recorded source_run_dir is
        # resolve()d by train(), and Drive mount points move.
        parts = Path(str(warm_start.source_run_dir)).parts[-2:]
        expected["warm_start"] = {
            "source_run_dir_suffix": "/".join(parts).lstrip("/"),
            "transfer_log_ent_coef": warm_start.transfer_log_ent_coef,
            "expected_artifact_sha256": warm_start.expected_artifact_sha256,
        }
    return expected


def run_leg(stage_name, *, seed, total_timesteps, warm_start=None):
    """Train one campaign leg in a fresh attempt dir; returns its run dir."""
    stage_dir = next_stage_attempt_dir(CAMPAIGN_ROOT, stage_name)
    print(f"\n===== {stage_name}: training into {stage_dir} =====")
    cfg = make_leg_config(
        log_dir=stage_dir,
        seed=seed,
        total_timesteps=total_timesteps,
        warm_start=warm_start,
    )
    model = train(cfg)
    del model  # the artifacts on disk are the campaign's interface
    # Prereg §6: validate the run's own config.json against the frozen
    # plan the leg launched under. The verdict lands in the manifest
    # either way -- before any gate scoring -- and a mismatch stops
    # the campaign rather than gating or branching a drifted run.
    expected = leg_expected_plan(
        seed=seed, total_timesteps=total_timesteps, warm_start=warm_start
    )
    validation = {"verdict": "ok"}
    try:
        config_path = locate_artifact(stage_dir, "config")
        if config_path is None:
            raise ValueError(f"no config.json found under {stage_dir}")
        validate_run_config_against_plan(config_path, expected)
    except ValueError as err:
        validation = {"verdict": "mismatch", "error": str(err)}
    manifest["stages"][stage_name] = {
        "status": (
            "trained" if validation["verdict"] == "ok" else "config_mismatch"
        ),
        "run_dir": str(stage_dir),
        "config_validation": validation,
    }
    write_campaign_manifest(CAMPAIGN_ROOT, manifest)
    if validation["verdict"] != "ok":
        raise ValueError(
            f"{stage_name}: the run's config.json does not match the "
            f"frozen plan -- {validation['error']}"
        )
    print(f"{stage_name}: config.json matches the frozen plan")
    print_stage_summary(stage_dir)
    summary_path = locate_artifact(stage_dir, "stage_summary")
    status_line = ""
    if summary_path is not None:
        with open(summary_path) as handle:
            status_line = next(
                (line for line in handle if line.startswith("Status:")), ""
            )
    if status_line.split(":", 1)[-1].strip() == "interrupted":
        raise RuntimeError(
            f"{stage_name} was interrupted; its artifacts are saved for "
            "recovery, but an interrupted leg cannot be gated or branched "
            "from. Re-run the notebook to retry it in a fresh attempt dir."
        )
    return stage_dir


def record_stage(stage_name, *, run_dir, report, extra=None):
    # Merge over what run_leg already recorded for this leg (the
    # config_validation verdict) instead of dropping it.
    record = dict(manifest["stages"].get(stage_name) or {})
    record.update(
        {
            "status": "complete",
            "run_dir": str(run_dir),
            "report": {
                "verdict": report["verdict"],
                "bars": {
                    name: bar["verdict"]
                    for name, bar in report["bars"].items()
                },
            },
        }
    )
    if extra:
        record.update(extra)
    manifest["stages"][stage_name] = record
    write_campaign_manifest(CAMPAIGN_ROOT, manifest)
    return record

## 6. Run the campaign — train, gate, branch, train

The whole campaign in one cell: leg 1 trains and is scored, the branch decision is made (or reused from the manifest), leg 2 warm-starts from the selected lineage and trains to its branch's budget. A `stop` decision ends the execution cleanly with instructions for resuming.

In [ ]:
# --- Leg 1: the from-scratch gate -----------------------------------
gate_record = manifest["stages"].get(GATE_STAGE)
if gate_record is None or gate_record["status"] != "complete":
    # Leg 1 is from scratch (the LS1 gate shape) unless the settings
    # pin a lineage -- the warm-started leg-1 path (e.g. an LT1-shape
    # temperature-skip pilot with its sha pins).
    leg1_warm_start = None
    if LEG1_WARM_START_RUN_DIR is not None:
        leg1_warm_start = WarmStartConfig(
            source_run_dir=str(LEG1_WARM_START_RUN_DIR),
            transfer_log_ent_coef=LEG1_TRANSFER_LOG_ENT_COEF,
            expected_artifact_sha256=LEG1_EXPECTED_ARTIFACT_SHA256,
        )
    gate_dir = run_leg(
        GATE_STAGE,
        seed=SEED,
        total_timesteps=GATE_TIMESTEPS,
        warm_start=leg1_warm_start,
    )
    gate_report = score_paddle_stage(
        gate_dir,
        bars=GATE_BARS,
        episodes=GATE_EPISODES,
        seed_start=GATE_SEED_START,
    )
    gate_record = record_stage(GATE_STAGE, run_dir=gate_dir, report=gate_report)
else:
    print(f"{GATE_STAGE} already complete: {gate_record['run_dir']}")

gate_verdict = gate_record["report"]["verdict"]
print(f"gate verdict: {gate_verdict} ({gate_record['report']['bars']})")

# --- Branch decision (recorded once; reused on resume) --------------
decision = manifest.get("branch")
if (
    decision is not None
    and decision["branch"] == "stop"
    and FORCE_BRANCH is not None
):
    # The maintainer decided a stopped gate; supersede the recorded stop.
    decision = None
if decision is None:
    if FORCE_BRANCH is not None:
        source = (
            gate_record["run_dir"]
            if FORCE_BRANCH == "continue"
            else FALLBACK_WARM_START_RUN_DIR
        )
        if source is None:
            raise ValueError(
                "FORCE_BRANCH='fallback' needs FALLBACK_WARM_START_RUN_DIR"
            )
        decision = {
            "branch": FORCE_BRANCH,
            "warm_start_source": str(source),
            "forced": True,
        }
    else:
        branch, source = resolve_warm_start_branch(
            gate_verdict,
            stage_run_dir=gate_record["run_dir"],
            fallback_run_dir=FALLBACK_WARM_START_RUN_DIR,
            middle_action=GATE_MIDDLE_ACTION,
        )
        decision = {
            "branch": branch,
            "warm_start_source": source,
            "forced": False,
        }
    manifest["branch"] = decision
    manifest["status"] = (
        "stopped_at_gate" if decision["branch"] == "stop" else "running"
    )
    write_campaign_manifest(CAMPAIGN_ROOT, manifest)
else:
    print(f"reusing recorded branch decision: {decision}")

# --- Leg 2: the main leg --------------------------------------------
if decision["branch"] == "stop":
    print(
        "STOP: the gate verdict ends the campaign here "
        f"(verdict {gate_verdict}, middle action {GATE_MIDDLE_ACTION!r}). "
        "Read the gate report, decide the branch, set FORCE_BRANCH, and "
        "re-run all cells to continue this campaign."
    )
else:
    main_record = manifest["stages"].get(MAIN_STAGE)
    if main_record is None or main_record["status"] != "complete":
        budget = (
            MAIN_TIMESTEPS_ON_PASS
            if decision["branch"] == "continue"
            else MAIN_TIMESTEPS_ON_FALLBACK
        )
        source = decision["warm_start_source"]
        if not Path(source).is_dir():
            raise FileNotFoundError(
                f"warm-start source is not a directory: {source}"
            )
        main_dir = run_leg(
            MAIN_STAGE,
            seed=SEED + 10_000,
            total_timesteps=budget,
            warm_start=WarmStartConfig(
                source_run_dir=str(source),
                transfer_log_ent_coef=LEG2_TRANSFER_LOG_ENT_COEF,
                expected_artifact_sha256=LEG2_EXPECTED_ARTIFACT_SHA256,
            ),
        )
        final_report = score_paddle_stage(
            main_dir,
            bars=FINAL_REPORT_BARS,
            episodes=GATE_EPISODES,
            seed_start=GATE_SEED_START,
            report_name="campaign_final_report.json",
        )
        main_record = record_stage(
            MAIN_STAGE,
            run_dir=main_dir,
            report=final_report,
            extra={"branch": decision["branch"], "budget": budget},
        )
    else:
        print(f"{MAIN_STAGE} already complete: {main_record['run_dir']}")
    manifest["status"] = "completed"
    write_campaign_manifest(CAMPAIGN_ROOT, manifest)

print("\ncampaign status:", manifest["status"])
print("manifest:", CAMPAIGN_ROOT / "campaign_manifest.json")

## 7. Reports & curves

Learning curves, eval-info pages, and training-health grids for every completed leg, saved into each leg's `reports/` folder on Drive.

In [ ]:
from courtside_dynamics.notebook_utils import (
    plot_eval_info,
    plot_learning_curve,
    plot_training_health,
)
from courtside_dynamics.training.artifacts import artifact_path

for stage_name, record in manifest["stages"].items():
    run_dir = record["run_dir"]
    print(f"\n===== {stage_name}: {run_dir} =====")
    print(f"report: {record['report']}")
    plot_learning_curve(
        run_dir, save_path=artifact_path(run_dir, "learning_curve")
    )
    plot_eval_info(run_dir, save_path=artifact_path(run_dir, "eval_headline"))
    plot_training_health(
        run_dir, save_path=artifact_path(run_dir, "training_health_plot")
    )

## 8. Artifact audit

Audits every completed leg against the shared artifact registry while the runtime still exists to fix anything missing.

In [ ]:
from courtside_dynamics.notebook_utils import check_run_artifacts

for stage_name, record in manifest["stages"].items():
    print(f"\n===== {stage_name} =====")
    missing = check_run_artifacts(record["run_dir"])
    if missing:
        print(f"{stage_name} missing artifacts: {missing}")

## 9. Disconnect Colab runtime

In [ ]:
# Frees the GPU when the campaign execution is over. Everything is
# already on Drive (manifest, run dirs, reports); comment out to keep
# the runtime for interactive inspection. No-op outside Colab.
from courtside_dynamics.notebook_utils import disconnect_runtime

if DISCONNECT_WHEN_DONE:
    disconnect_runtime(delay_seconds=30)